## Extraction des videos et playliste (Chaine youtube : Wandaloo)

In [8]:
import requests
import json
import os
from dotenv import load_dotenv
from datetime import datetime

# Chargement de l'environnement
load_dotenv(encoding='utf-16')
API_KEY = os.getenv("YOUTUBE_API_KEY")
HANDLE = "@wandaloo" 

def get_channel_id_from_handle(handle):
    """Convertit un handle (@nom) en Channel ID via l'API YouTube."""
    clean_handle = handle.replace('@', '')
    url = f"https://youtube.googleapis.com/youtube/v3/channels?part=id&forHandle={clean_handle}&key={API_KEY}"
    
    response = requests.get(url).json()
    items = response.get('items', [])
    
    if not items:
        raise ValueError(f"Impossible de trouver la chaîne pour le handle : {handle}")
    
    return items[0]['id']

def get_wandaloo_separated_data():
    # --- ÉTAPE 0 : RÉSOLUTION DU ID ---
    print(f"Étape 0 : Résolution de l'ID pour le handle {HANDLE}...")
    CHANNEL_ID = get_channel_id_from_handle(HANDLE)
    print(f"ID trouvé : {CHANNEL_ID}")

    # --- 1. EXTRACTION DES PLAYLISTS ---
    print("Étape 1 : Extraction des métadonnées des playlists...")
    pl_url = f"https://youtube.googleapis.com/youtube/v3/playlists?part=snippet,contentDetails&channelId={CHANNEL_ID}&maxResults=50&key={API_KEY}"
    playlists_raw = requests.get(pl_url).json().get('items', [])
    
    playlists_metadata = []
    video_to_pl_map = {} 
    
    for pl in playlists_raw:
        pl_id = pl['id']
        playlists_metadata.append({
            "playlist_id": pl_id,
            "playlist_title": pl['snippet']['title'],
            "playlist_description": pl['snippet']['description'],
            "playlist_published_at": pl['snippet']['publishedAt'],
            "total_videos_in_playlist": pl['contentDetails']['itemCount'],
            "playlist_url": f"https://www.youtube.com/playlist?list={pl_id}"
        })
        
        next_page = None
        while True:
            it_url = f"https://youtube.googleapis.com/youtube/v3/playlistItems?part=contentDetails&playlistId={pl_id}&maxResults=50&key={API_KEY}"
            if next_page: it_url += f"&pageToken={next_page}"
            res = requests.get(it_url).json()
            for item in res.get('items', []):
                v_id = item['contentDetails']['videoId']
                if v_id not in video_to_pl_map:
                    video_to_pl_map[v_id] = pl_id
            next_page = res.get('nextPageToken')
            if not next_page: break

    # --- 2. EXTRACTION DE TOUTES LES VIDÉOS ---
    print("Étape 2 : Extraction de la totalité des vidéos...")
    all_videos = []
    ch_url = f"https://youtube.googleapis.com/youtube/v3/channels?part=contentDetails&id={CHANNEL_ID}&key={API_KEY}"
    uploads_id = requests.get(ch_url).json()['items'][0]['contentDetails']['relatedPlaylists']['uploads']
    
    next_page = None
    while True:
        url = f"https://youtube.googleapis.com/youtube/v3/playlistItems?part=snippet,contentDetails&playlistId={uploads_id}&maxResults=50&key={API_KEY}"
        if next_page: url += f"&pageToken={next_page}"
        
        res = requests.get(url).json()
        batch_ids = [item['contentDetails']['videoId'] for item in res.get('items', [])]
        
        if batch_ids:
            v_url = f"https://youtube.googleapis.com/youtube/v3/videos?part=snippet,contentDetails,statistics&id={','.join(batch_ids)}&key={API_KEY}"
            v_res = requests.get(v_url).json()
            
            for v in v_res.get('items', []):
                v_id = v['id']
                all_videos.append({
                    "videoId": v_id,
                    "playlist_id": video_to_pl_map.get(v_id, "OFF_PLAYLIST"),
                    "title": v['snippet']['title'],
                    "publishedAt": v['snippet']['publishedAt'],
                    "duration": v['contentDetails']['duration'],
                    "viewCount": int(v['statistics'].get('viewCount', 0)),
                    "likeCount": int(v['statistics'].get('likeCount', 0)),
                    "commentCount": int(v['statistics'].get('commentCount', 0))
                })
        
        next_page = res.get('nextPageToken')
        if not next_page: break

    # --- 3. SAUVEGARDE AVEC HORODATAGE ---
    os.makedirs('data', exist_ok=True)
    
    # Création du timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    video_filename = f'data/wandaloo_videos_{timestamp}.json'
    playlist_filename = f'data/wandaloo_playlists_{timestamp}.json'

    with open(video_filename, 'w', encoding='utf-8') as f:
        json.dump(all_videos, f, ensure_ascii=False, indent=4)
        
    with open(playlist_filename, 'w', encoding='utf-8') as f:
        json.dump(playlists_metadata, f, ensure_ascii=False, indent=4)
    
    print(f"✅ Extraction terminée pour {HANDLE} !")
    print(f"-> Fichiers créés : {video_filename} et {playlist_filename}")

if __name__ == "__main__":
    get_wandaloo_separated_data()

Étape 0 : Résolution de l'ID pour le handle @wandaloo...
ID trouvé : UCSJeKLv7SkvcMuaWL4VL8Ig
Étape 1 : Extraction des métadonnées des playlists...
Étape 2 : Extraction de la totalité des vidéos...
✅ Extraction terminée pour @wandaloo !
-> Fichiers créés : data/wandaloo_videos_20260503_014023.json et data/wandaloo_playlists_20260503_014023.json


## Transformation et structuration des données

In [9]:
import json
import pandas as pd
import re
import os
from datetime import timedelta, datetime

def parse_duration(duration_str):
    """Convertit une durée ISO 8601 (ex: PT2M55S) en secondes totales."""
    if not duration_str or pd.isna(duration_str):
        return 0
    pattern = re.compile(r'PT(?:(?P<hours>\d+)H)?(?:(?P<minutes>\d+)M)?(?:(?P<seconds>\d+)S)?')
    match = pattern.match(duration_str)
    if not match:
        return 0
    parts = match.groupdict()
    time_params = {name: int(num) for name, num in parts.items() if num}
    return int(timedelta(**time_params).total_seconds())

def transform_data(video_json_file, playlist_json_file):
    # --- 1. CONFIGURATION & MARQUES 
    brands = [
        "ALFA ROMEO", "AUDI", "BAIC", "BMW", "BYD", "CHANGAN", "CHERY",
        "CITROËN","Citroën", "JAGUAR","CITROËN","HYUNDAI", "CUPRA", "DACIA","DEEPAL", "DONGFENG", "DS", "EXEED", 
        "FIAT", "FORD", "GEELY", "GWM", "TANK", "HAVAL", "HONDA", "HYUNDAI", "JAC", 
        "JAECOO", "JAGUAR", "JETOUR", "KGM", "SSANGYONG", "KIA", "LEAPMOTOR", "LEXUS", 
        "LYNK & CO", "MAHINDRA", "MERCEDES-BENZ", "MERCESDES-Benz", "MERCEDES", "MG", "NISSAN", 
        "OPEL", "PEUGEOT", "PORSCHE", "RENAULT", "ROX", "SEAT", "ŠKODA",'SKODA','Škoda',   
        "SMART", "SOUEAST", "SUZUKI", "TOYOTA", "VOLKSWAGEN", "VW", "VOLVO", 
        "XIAOMI", "ZEEKR",'XPENG','RANGE ROVER','JEEP','TOGG','LAND ROVER','MINI','CHEVROLET' 
    ]

    # --- 2. TRAITEMENT DES PLAYLISTS ---
    with open(playlist_json_file, 'r', encoding='utf-8') as f:
        df_playlists = pd.DataFrame(json.load(f))

    df_playlists['playlist_published_at'] = pd.to_datetime(df_playlists['playlist_published_at'])
    df_playlists['total_videos_in_playlist'] = pd.to_numeric(df_playlists['total_videos_in_playlist'], errors='coerce').fillna(0).astype(int)

    # --- 3. TRAITEMENT DES VIDÉOS ---
    with open(video_json_file, 'r', encoding='utf-8') as f:
        videos_data = json.load(f)

    processed_videos = []
    for v in videos_data:
        title = v.get('title', '')
        # On passe le titre en majuscule pour la comparaison avec la liste brands
        title_upper = title.upper()
        
        brand_found = "AUTRE / MULTI-MARQUES"
        for b in brands:
            if re.search(rf'\b{b}\b', title_upper):
                brand_found = b
                break

        processed_videos.append({
            'videoId': v.get('videoId'),
            'playlist_id': v.get('playlist_id', 'OFF_PLAYLIST'),
            'brand': brand_found,
            'title': title,
            'viewCount': v.get('viewCount', 0),
            'likeCount': v.get('likeCount', 0),
            'commentCount': v.get('commentCount', 0),
            'publishedAt': v.get('publishedAt'),
            'duration_seconds': parse_duration(v.get('duration'))
        })

    df_videos = pd.DataFrame(processed_videos)

    # --- 4. NETTOYAGE FINAL ---
    df_videos['publishedAt'] = pd.to_datetime(df_videos['publishedAt'])
    numeric_cols = ['viewCount', 'likeCount', 'commentCount']
    for col in numeric_cols:
        df_videos[col] = pd.to_numeric(df_videos[col], errors='coerce').fillna(0).astype(int)

    # --- 5. STOCKAGE AVEC DATE ET HEURE ---
    if not os.path.exists('data'): 
        os.makedirs('data')

    # Génération du timestamp (ex: 20260503_143005)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    file_name_playlists = f'data/dim_playlists_{timestamp}.csv'
    file_name_videos = f'data/fact_videos_{timestamp}.csv'

    df_playlists.to_csv(file_name_playlists, index=False, encoding='utf-8-sig')
    df_videos.to_csv(file_name_videos, index=False, encoding='utf-8-sig')
    
    print(f"Normalisation terminée.")
    print(f"-> Fichier playlists : {file_name_playlists}")
    print(f"-> Fichier vidéos : {file_name_videos}")

if __name__ == "__main__":
    transform_data('data/wandaloo_videos_20260503_014023.json', 'data/wandaloo_playlists_20260503_014023.json')

Normalisation terminée.
-> Fichier playlists : data/dim_playlists_20260503_014524.csv
-> Fichier vidéos : data/fact_videos_20260503_014524.csv


In [12]:
import pandas as pd

df1=pd.read_csv('data/fact_videos_20260503_014524.csv')
df1.head()

,videoId,playlist_id,brand,title,viewCount,likeCount,commentCount,publishedAt,duration_seconds
0,KJZs0NouLwg,PLlvsaB0FDEJGqNWcPIvRM1Fex7rj7cyxz,JEEP,JEEP Compass 2026 : lancement au Maroc - الإطل...,2067,19,5,2026-04-30 21:45:06+00:00,259
1,jhQzy3m0EQ4,PLlvsaB0FDEJG0eoTrNpNKZXLLlXKiEmE-,VW,VW ID Polo 2027 : le spot officiel,1070,6,0,2026-04-30 20:32:30+00:00,127
2,azQYAsSjKEM,PLlvsaB0FDEJGqNWcPIvRM1Fex7rj7cyxz,GWM,GWM Tank 300 vainqueur Rallye Aïcha des Gazell...,872,4,0,2026-04-29 23:30:06+00:00,175
3,0LzvCQFQ4-0,PLlvsaB0FDEJGqNWcPIvRM1Fex7rj7cyxz,LEXUS,LEXUS RZ 550e : le spot officiel,1433,9,0,2026-04-29 23:15:01+00:00,77
4,Cnl_b0afMqM,PLlvsaB0FDEJF6Od1WVIrvaNBNdnAi1-o_,ALFA ROMEO,ALFA ROMEO célèbre son 115ème anniversaire,81,3,0,2026-04-29 23:00:13+00:00,51


In [13]:
df2=pd.read_csv('data/dim_playlists_20260503_014524.csv')
df2.head()  

,playlist_id,playlist_title,playlist_description,playlist_published_at,total_videos_in_playlist,playlist_url
0,PLlvsaB0FDEJEIQJVysOtRwAwe2CgMc98S,OCCASION,NaN,2022-03-16 19:01:38.218637+00:00,13,https://www.youtube.com/playlist?list=PLlvsaB0...
1,PLlvsaB0FDEJFZxc1DFcxPny2lFNj6djgG,EURO NCAP,NaN,2021-12-08 10:37:35.220377+00:00,12,https://www.youtube.com/playlist?list=PLlvsaB0...
2,PLlvsaB0FDEJF4DIiRHIaKTNT1O8hkoe2_,Salon de Genève,Découvrez en vidéo les principales nouveautés ...,2020-03-03 18:58:31.517317+00:00,41,https://www.youtube.com/playlist?list=PLlvsaB0...
3,PLlvsaB0FDEJFNBfe474lsoxan7tHM37RE,Salon de Francfort,NaN,2019-09-09 17:37:03.703001+00:00,6,https://www.youtube.com/playlist?list=PLlvsaB0...
4,PLlvsaB0FDEJG4o9DNYvXjEFh4GW1Am_Ow,Salon de Paris,Découvrez en vidéo les principales nouveautés ...,2018-10-04 18:15:54.253609+00:00,43,https://www.youtube.com/playlist?list=PLlvsaB0...
